# [3.1] SC26 Tutorial — Session 2, Part 1: Linalg and tiling (CPU)

Session 1 built a matrix-matrix product from the bottom: `scf.for` loops written by hand, then a custom `blas.matmul` op with a pass that lowered it to those same loops.

This session introduces operations in the Linalg dialect, which models a class of linear algebra operations using techniques from the polyhedral compilation domain.

This part:

- introduces the `linalg.matvec` and `linalg.matmul` operations, and the `LinalgOp` interface, which is used to write transformations that are valid for all operations in the dialect;
- uses the transform dialect to apply a tiling transformation to Linalg operations.

Everything in this part runs on CPU only, single-threaded.

**Running locally in VS Code instead?** See [`RUNNING_LOCALLY.md`](https://github.com/MaheshRavishankar/sc26_tutorial/blob/main/RUNNING_LOCALLY.md).

## [3.2] Setup

**Run the cells in order, starting here.** Later cells use names defined by earlier ones, so skipping ahead can lead to strange errors. To catch up after losing track, `Ctrl+F8` on Colab runs every cell above the current one.

The next cells download the same prebuilt MLIR toolchain as Session 1.

In [ ]:
# [3.3]
REPO = "MaheshRavishankar/sc26_tutorial"
RELEASE_TAG = "session1-toolchain-v1"
ASSET_NAME = "mlir-tutorial-llvm.tar.gz"

In [ ]:
# [3.4]
import os

# Set MLIR_TUTORIAL_INSTALL_DIR to iterate locally (e.g. in VS Code) against
# an already-built toolchain -- skips the download entirely if it's already
# there. Unset in Colab, so this always defaults to the normal /content path.
INSTALL_DIR = os.environ.get("MLIR_TUTORIAL_INSTALL_DIR", "/content/mlir-tutorial-llvm")

if os.path.exists(f"{INSTALL_DIR}/bin/mlir-opt"):
    print(f"Using existing toolchain at {INSTALL_DIR}")
else:
    import requests

    tar_path = f"{INSTALL_DIR}.tar.gz"
    url = f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(tar_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

    !mkdir -p {INSTALL_DIR}
    !tar xf {tar_path} -C {INSTALL_DIR} --strip-components=1

In [ ]:
# [3.5]
import glob, os, re, sys

os.environ["PATH"] = f"{INSTALL_DIR}/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{INSTALL_DIR}/python_packages/mlir_core")

# The MLIR Python bindings only import into the Python version they were built
# for (3.13, as on Colab). Check that here: with any other version, the next
# cell fails with a confusing "cannot import name 'ir'".
bindings = glob.glob(f"{INSTALL_DIR}/python_packages/mlir_core/mlir/_mlir_libs/_mlir.cpython-*.so")
assert bindings, f"no MLIR Python bindings in {INSTALL_DIR}"
built_for = ".".join(re.search(r"cpython-(\d)(\d+)", bindings[0]).groups())
kernel = f"{sys.version_info.major}.{sys.version_info.minor}"
assert built_for == kernel, (
    f"The MLIR Python bindings are built for Python {built_for}; this kernel is "
    f"Python {kernel}. Switch to a Python {built_for} kernel.")

!mlir-opt --version

In [ ]:
# [3.6]
from mlir.ir import Context, Module
from mlir.passmanager import PassManager
from mlir.execution_engine import ExecutionEngine
import mlir.runtime as rt
import ctypes
import numpy as np

print("MLIR Python bindings imported OK")

**[3.7]** The next cell clones the tutorial's repo.

In [ ]:
# [3.8]
# Set MLIR_TUTORIAL_REPO_DIR to iterate locally against an already-cloned
# checkout -- skips the clone entirely if it's already there.
REPO_DIR = os.environ.get("MLIR_TUTORIAL_REPO_DIR", "/content/sc26_tutorial")
if os.path.isdir(REPO_DIR):
    print(f"Using existing repo checkout at {REPO_DIR}")
else:
    # --no-single-branch: every branch's tip, still at depth 1. It costs
    # nothing measurable, and it means the solution branches are already in
    # the checkout, for taking a solved file out of one.
    !git clone --depth 1 --no-single-branch https://github.com/{REPO} {REPO_DIR}

## [3.9] Step 1 — Linalg ops

`session2/linalg_matmul.mlir` and `session2/linalg_matvec.mlir` hold one op each: `linalg.matmul`, which computes `c += a @ b`, and `linalg.matvec`.

In [ ]:
# [3.10]
# Opens the files. On Colab, files.view() puts them in the editor pane -- use it
# instead of the file browser, which refuses some file types. Running locally
# there is no google.colab, so this prints the paths instead, to open in an
# editor.
MATMUL_FILE = f"{REPO_DIR}/session2/linalg_matmul.mlir"
MATVEC_FILE = f"{REPO_DIR}/session2/linalg_matvec.mlir"

try:
    from google.colab import files

    files.view(MATMUL_FILE)
    files.view(MATVEC_FILE)
except ModuleNotFoundError:
    print(f"open in an editor:\n    {MATMUL_FILE}\n    {MATVEC_FILE}")

**[3.11]** Round-trip both through `mlir-opt` first. No pass pipeline, so this only parses and re-prints each file.

In [ ]:
# [3.12]
!mlir-opt {MATMUL_FILE}
assert _exit_code == 0
!mlir-opt {MATVEC_FILE}
assert _exit_code == 0

**[3.13]** Both ops implement the `LinalgOp` interface, which requires each op to have:

- **`iterator_types`** — one per loop, `parallel` or `reduction`. Matmul has three: `i` and `j` parallel, `k` the reduction. Matvec has two: `i` parallel, `j` the reduction.
- **`indexing_maps`** — one per operand, saying which loop indices address it. For matmul, `a` is read at `(i, k)`, `b` at `(k, j)`, and `c` at `(i, j)`.
- **the body** — the scalar computation at each iteration: a `mulf`, then an `addf` into the output element.

The interface also verifies each op: using the indexing maps, it checks that the operands agree on the size of every loop. That is a general version of the dimension checks in `blas.matmul`'s verifier from Session 1 (Exercise 2b).

`linalg.matmul` and `linalg.matvec` are operations whose iterator types, indexing maps and body are fixed to give matmul and matvec semantics. `linalg.generic` takes all three explicitly. The `linalg-generalize-named-ops` pass, run below, rewrites both ops as `linalg.generic`, which shows what each one fixes.

In [ ]:
# [3.14]
!mlir-opt {MATMUL_FILE} --linalg-generalize-named-ops
assert _exit_code == 0
!mlir-opt {MATVEC_FILE} --linalg-generalize-named-ops
assert _exit_code == 0

**[3.15]** `convert-linalg-to-loops` replaces each op with the loop nest it stands for: one `scf.for` per iterator type, a `memref.load` of each operand at the indices its indexing map selects, the body, and a `memref.store` back into the output. Compare `@matmul` against the `matmul_scf.mlir` written by hand in Session 1 — it is the same nest, generated rather than typed.

In [ ]:
# [3.16]
!mlir-opt {MATMUL_FILE} --convert-linalg-to-loops
assert _exit_code == 0
!mlir-opt {MATVEC_FILE} --convert-linalg-to-loops
assert _exit_code == 0

**[3.17]** Lower the rest of the way to the LLVM dialect and JIT `@matmul`, checking it against numpy. The pipeline is Session 1's seven passes with `convert-linalg-to-loops` in front of them: it does the job the `convert-blas-to-scf` pass did there.

In [ ]:
# [3.18]
CPU_LOWERING_PIPELINE = (
    "builtin.module("
    "func.func(llvm-request-c-wrappers),"
    "func.func(convert-linalg-to-loops),"
    "convert-scf-to-cf,"
    "convert-cf-to-llvm,"
    "convert-arith-to-llvm,"
    "finalize-memref-to-llvm,"
    "convert-func-to-llvm,"
    "reconcile-unrealized-casts"
    ")"
)

M, N, K = 128, 256, 512  # c is M x N; a is M x K; b is K x N


def memref_args(*arrays):
    return [ctypes.pointer(ctypes.pointer(rt.get_ranked_memref_descriptor(x))) for x in arrays]


def compile_file(path, **engine_options):
    """Parse an MLIR file, lower it to LLVM, and JIT-compile it."""
    ctx = Context()
    module = Module.parseFile(path, ctx)
    PassManager.parse(CPU_LOWERING_PIPELINE, ctx).run(module.operation)
    return ExecutionEngine(module, opt_level=3, **engine_options)


def run_and_check(engine, args, expected):
    """Call `matmul` on numpy arrays passed as memrefs. The last one is the
    output buffer, which linalg.matmul accumulates into, so it is zeroed
    first."""
    args[-1][...] = 0
    engine.invoke("matmul", *memref_args(*args))
    err = float(np.max(np.abs(args[-1] - expected)))
    print(f"result {args[-1].shape}, max abs error vs numpy = {err:.3g}")
    np.testing.assert_allclose(args[-1], expected, rtol=1e-3, atol=1e-3)
    print("OK")


rng = np.random.default_rng(0)
a = rng.random((M, K), dtype=np.float32)
b = rng.random((K, N), dtype=np.float32)
c = np.zeros((M, N), dtype=np.float32)
expected = a @ b

untiled_matmul = compile_file(MATMUL_FILE)
run_and_check(untiled_matmul, [a, b, c], expected)

## [3.19] Step 2 — tiling, with a transform dialect schedule

**Tiling** splits a loop nest into two sets of loops: outer loops that iterate over tiles, and inner loops that do the computation within a tile. It is the same computation, in a different order.

The tiling is written in the **transform dialect**, which specifies transformations on operations and applies them. Its operations are transformations to apply; its values are handles to operations in the program being transformed, and an operation's results are handles to the operations it creates. Changes to the transformations are made by editing the transform dialect file: they need no change to C++ code and no rebuild of the compiler.

**`session2/tile_linalg.mlir`** is the schedule. It matches ops by the `LinalgOp` interface, so it finds both ops, and tiles them with `tile_using_for` and tile sizes `[32, 32]`, one per loop in the order of `iterator_types`. That tiles both of matvec's loops, but only `i` and `j` of matmul's three.

In [ ]:
# [3.20]
# Opens the file to edit. On Colab, files.view() puts it in the editor pane.
# Running locally there is no google.colab, so this prints the path instead,
# to open in an editor.
SCHEDULE = f"{REPO_DIR}/session2/tile_linalg.mlir"

try:
    from google.colab import files

    files.view(SCHEDULE)
except ModuleNotFoundError:
    print(f"open in an editor -- the file to edit:\n    {SCHEDULE}")

**[3.21]** Like the operations of any other MLIR dialect, transform dialect operations can be round-tripped through `mlir-opt` to check that the schedule is specified correctly.

In [ ]:
# [3.22]
!mlir-opt {SCHEDULE}
assert _exit_code == 0

**[3.23]** Apply the schedule to matvec first. `--transform-preload-library` loads the schedule, and `--transform-interpreter` finds `@__transform_main` in it and applies it to the file.

Tiling a Linalg operation produces `scf.for` loops over the tiles, but the computation within the loops is still expressed as the same Linalg operation, working on `memref.subview` slices of its operands.

In [ ]:
# [3.24]
MATVEC_TILED = "linalg_matvec_tiled.mlir"

!mlir-opt {MATVEC_FILE} \
    --transform-preload-library=transform-library-paths={SCHEDULE} \
    --transform-interpreter \
    -o {MATVEC_TILED}
assert _exit_code == 0
!cat {MATVEC_TILED}

assert "memref.subview" in open(MATVEC_TILED).read(), "@matvec was not tiled"

### [3.25] Tiling matmul's `k` loop

**Exercise 5: extend `tile_linalg.mlir` so that matmul's `k` loop is tiled by 32 as well.**

- `transform.structured.match ops{["linalg.matmul"]}` matches by op name instead of by interface.
- A tile size of `0` leaves that loop untiled.

**Then run the cells below one at a time.** They round-trip the schedule, apply it to matmul and check the result. After each edit, re-run them from the round-trip on.

In [ ]:
# [3.26]
!mlir-opt {SCHEDULE}
assert _exit_code == 0

**[3.27]** Apply the schedule to matmul. As provided, the schedule tiles only `i` and `j`, so the innermost `linalg.matmul` works on 32x512 slices of `a` and 512x32 slices of `b`. Once Exercise 5 is done, every slice is 32x32.

In [ ]:
# [3.28]
MATMUL_TILED = "linalg_matmul_tiled.mlir"

!mlir-opt {MATMUL_FILE} \
    --transform-preload-library=transform-library-paths={SCHEDULE} \
    --transform-interpreter \
    -o {MATMUL_TILED}
assert _exit_code == 0
!cat {MATMUL_TILED}

**[3.29]** **The check on Exercise 5** passes when every operand of the innermost `linalg.matmul` is a 32x32 slice, so all three loops are tiled.

In [ ]:
# [3.30]
import re

matmul_line = next(line for line in open(MATMUL_TILED) if "linalg.matmul" in line)
operand_shapes = re.findall(r"memref<(\d+x\d+)xf32", matmul_line)
print("shapes the innermost linalg.matmul works on:", operand_shapes)
assert operand_shapes and all(shape == "32x32" for shape in operand_shapes), (
    "@matmul is not tiled in all three loops yet -- a 512 in these shapes is a "
    "loop that still runs untiled")

**[3.31]** Now run the tiled matmul, through the same numpy check as step 1.

The lowering needs two more passes this time, for the ops tiling introduced:

- **`expand-strided-metadata`** breaks each `memref.subview` into raw pointer, offset and stride arithmetic.
- **`lower-affine`** turns that arithmetic, which comes out as `affine.apply` ops, into plain `arith`.

The cell below adds both to `CPU_LOWERING_PIPELINE`, right after `convert-linalg-to-loops`.

In [ ]:
# [3.32]
# Tiling introduced memref.subview: add the two passes that lower it.
CPU_LOWERING_PIPELINE = (
    "builtin.module("
    "func.func(llvm-request-c-wrappers),"
    "func.func(convert-linalg-to-loops),"
    "expand-strided-metadata,"
    "lower-affine,"
    "convert-scf-to-cf,"
    "convert-cf-to-llvm,"
    "convert-arith-to-llvm,"
    "finalize-memref-to-llvm,"
    "convert-func-to-llvm,"
    "reconcile-unrealized-casts"
    ")"
)

tiled_matmul = compile_file(MATMUL_TILED)
run_and_check(tiled_matmul, [a, b, c], expected)

## [3.33] Step 3 — did matmul get faster?

Same computation, same numbers, same single thread. Time the untiled and tiled matmul and see.

`benchmark` keeps the **fastest** of several runs rather than the average. Colab is a shared machine, so a slow run mostly measures other users.

In [ ]:
# [3.34]
import time


def benchmark(name, engine, reps=5):
    best = float("inf")
    for _ in range(reps):
        c[...] = 0
        args = memref_args(a, b, c)
        start = time.perf_counter()
        engine.invoke("matmul", *args)
        best = min(best, time.perf_counter() - start)
    gflops = 2 * M * N * K / best / 1e9  # two float ops per innermost iteration
    print(f"{name:>10}: {best * 1000:8.1f} ms   {gflops:6.2f} GFLOP/s")
    return best


untiled_time = benchmark("untiled", untiled_matmul)
tiled_time = benchmark("tiled", tiled_matmul)
print(f"\nspeedup from tiling: {untiled_time / tiled_time:.1f}x")

### [3.35] Why so much faster?

The usual explanation for tiling is the cache. Here the three matrices together are under 1 MB, so they largely fit in cache untiled.

What changed is the **machine code LLVM generated**. The cell below asks the JIT to dump the object code it produced for each version and counts the floating-point multiply and add instructions in `matmul`:

- **scalar** instructions (`mulss`, `addss`, ...) work on one number at a time;
- **packed** instructions (`mulps`, `addps`, ...) work on a whole vector of 4, 8 or 16 numbers at once, in the `xmm`, `ymm` or `zmm` registers.

In [ ]:
# [3.36]
import shutil

# objdump comes with binutils. Install it only where it is missing, so this
# also runs locally, where apt-get would need root.
if shutil.which("objdump"):
    print("objdump is already installed")
else:
    !apt-get install -y -qq binutils > /dev/null
    assert _exit_code == 0

In [ ]:
# [3.37]
import shutil
import subprocess


def float_ops(path):
    """JIT-compile `path`, dump its object code, and count the float multiply
    and add instructions in `matmul`: scalar (...ss) and packed (...ps)."""
    engine = compile_file(path, enable_object_dump=True)
    obj = os.path.basename(path).replace(".mlir", ".o")
    engine.dump_to_object_file(obj)
    asm = subprocess.run(["objdump", "-d", "--no-show-raw-insn", obj],
                         capture_output=True, text=True).stdout
    body = asm.split("<matmul>:", 1)[1].split("\n\n", 1)[0]
    ops = re.findall(r"\s(v?(?:mul|add|fmadd\w*)(ss|ps))\s+(\S+)", body)
    scalar = sum(1 for _, kind, _ in ops if kind == "ss")
    packed = [operands for _, kind, operands in ops if kind == "ps"]
    width = {"zmm": 16, "ymm": 8, "xmm": 4}
    lanes = max((w for reg, w in width.items() if any(reg in p for p in packed)), default=0)
    return scalar, len(packed), lanes


if shutil.which("objdump") is None:
    print("objdump is not installed here -- `apt-get install binutils`, or skip this cell.")
else:
    print(f"{'':>8} {'scalar ops':>11} {'packed ops':>11}  vector width")
    for label, path in (("untiled", MATMUL_FILE), ("tiled", MATMUL_TILED)):
        scalar, packed, lanes = float_ops(path)
        print(f"{label:>8} {scalar:>11} {packed:>11}  {f'{lanes} floats at once' if packed else '-'}")

**[3.38]** **The speedup is vectorization, and tiling is what made it possible.**

1. **Untiled, the innermost loop is the reduction** `c[i,j] += a[i,k] * b[k,j]` over `k`. A running float sum cannot be vectorized without changing the order of the additions, which changes the answer in its last bits, and LLVM will not do that unless told it may. So the untiled matmul multiplies one pair of numbers at a time.
2. **Tiled, `k` runs a small, constant 32 times inside each tile.** LLVM *unrolls* a loop like that completely: it pastes the body out 32 times as straight-line code, and the `k` loop disappears.
3. **What is left around it, the loop over `j`, is a plain loop whose iterations are independent.** That loop *can* be vectorized, so several `c[i,j]` are computed at once. That is the packed column above.

Both steps are decisions made by cost models inside LLVM, which is why this session fixes the tile size at 32. With LLVM 22 on every x86 CPU Colab hands out, tiles of 16 or 32 get through both steps; tiles of 24 or 64 do not, and the kernel stays scalar.